# LeetCode 3750 — Minimum Number of Flips to Reverse Binary String bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-flips-to-reverse-binary-string/ · Độ khó: **Easy** · Chủ đề: Math, Two Pointers, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def minimumFlips(self, n: int) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org / WebSearch (không tìm thấy trang leetcode.com trong kết quả, nên slug `minimum-number-of-flips-to-reverse-binary-string` suy ra từ tên bài). Hai ví dụ `n = 7 -> 0` và `n = 10 -> 4` khớp với cách hiểu bên dưới; nếu có điều kiện, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đổi `n` thành chuỗi nhị phân `s` (không có số `0` đứng đầu); mỗi lần được **lật một bit**; tìm số lần lật ít nhất để `s` trở thành **chính chuỗi đảo ngược của `s` ban đầu**.
2. Phân biệt bài này với "biến `s` thành palindrome": ở đây đích là `reverse(s)` cố định nên mỗi vị trí lệch phải được sửa **riêng lẻ**.
3. Dùng `bin(n)[2:]`, lát cắt đảo `s[::-1]`, `zip`, và hai con trỏ để so sánh đầu-cuối.
4. Viết 4 cách: so sánh `s` với `s[::-1]` từng vị trí, hai con trỏ đếm cặp lệch rồi nhân đôi, thao tác bit thuần (không chuỗi), và XOR với số đảo bit rồi đếm bit `1`; đối chiếu bằng test ngẫu nhiên có `random.seed` và một oracle vét cạn theo mặt nạ lật.
5. Tránh các lỗi: quên nhân đôi, đếm mỗi cặp hai lần rồi nhân đôi tiếp, quên bỏ tiền tố `'0b'`, và dùng hiệu hai số thay vì số bit khác nhau.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`. Gọi `s` là biểu diễn nhị phân của `n` **không có số `0` ở đầu**. Bạn được phép **lật** một bit bất kỳ của `s` (`0 -> 1` hoặc `1 -> 0`); mỗi lần lật tính một thao tác. Hãy trả về số thao tác **ít nhất** để `s` trở thành **bản đảo ngược của chính nó lúc ban đầu** (`reverse` của `s` ban đầu).

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `7` | `0` | `s = "111"`, đảo ngược cũng là `"111"` nên không cần lật |
| 2 | `10` | `4` | `s = "1010"`, đảo ngược là `"0101"`: cả bốn bit đều khác, phải lật cả 4 |

Constraints: `1 <= n <= 10^9`.

In [1]:
for n in [7, 10]:
    s = bin(n)[2:]
    r = s[::-1]
    diff = [i for i in range(len(s)) if s[i] != r[i]]
    print(f"n={n} s={s} đảo={r} vị trí khác nhau={diff} -> số lần lật = {len(diff)}")

n=7 s=111 đảo=111 vị trí khác nhau=[] -> số lần lật = 0
n=10 s=1010 đảo=0101 vị trí khác nhau=[0, 1, 2, 3] -> số lần lật = 4


## 3. Kiến thức nền

### 3.1. `bin()` và chuỗi nhị phân

`bin(10)` trả về `'0b1010'`; tiền tố `'0b'` chỉ là ký hiệu, nên ta lấy `bin(n)[2:]` để có chữ số nhị phân thực sự. Chuỗi này **không** có `0` đứng đầu (trừ khi `n = 0`). Cũng có thể dùng `format(n, 'b')`.

In [2]:
print(bin(10), bin(10)[2:], format(10, 'b'), bin(7)[2:])
print(int("1010", 2), int("0101", 2))      # đổi ngược chuỗi nhị phân -> số

0b1010 1010 1010 111
10 5


### 3.2. Lát cắt đảo `s[::-1]` và `zip`

`s[::-1]` là chuỗi đảo ngược. `zip(s, s[::-1])` ghép ký tự ở vị trí `i` với ký tự ở vị trí `L-1-i`; hay dùng để so hai chuỗi cùng độ dài.

In [3]:
s = "1010"
print(s[::-1])
print(list(zip(s, s[::-1])))
print(sum(a != b for a, b in zip(s, s[::-1])))       # số vị trí khác nhau

0101
[('1', '0'), ('0', '1'), ('1', '0'), ('0', '1')]
4


### 3.3. Hai con trỏ từ hai đầu

Với `i = 0`, `j = L - 1` rồi `i += 1; j -= 1` cho đến khi `i >= j` ta duyệt đúng các **cặp đối xứng** `(s[i], s[L-1-i])`. Ký tự ở giữa (khi `L` lẻ) tự ghép với chính nó nên không bao giờ lệch.

In [4]:
s = "10110"
i, j = 0, len(s) - 1
while i < j:
    print(f"cặp ({i},{j}): {s[i]} và {s[j]} -> {'lệch' if s[i] != s[j] else 'khớp'}")
    i += 1
    j -= 1

cặp (0,4): 1 và 0 -> lệch
cặp (1,3): 0 và 1 -> lệch


### 3.4. Toán tử bit: `>>`, `&`, `^`, `bit_length`

- `n >> i` dịch phải `i` bit; `(n >> i) & 1` lấy bit thứ `i` (tính từ bit thấp nhất, bắt đầu `0`).
- `a ^ b` (XOR) cho bit `1` ở những vị trí hai số **khác nhau**.
- `n.bit_length()` là số bit của `n` (độ dài chuỗi nhị phân).
- `bin(x).count('1')` đếm số bit `1` (popcount).

In [5]:
n = 10                              # 1010
print([(n >> i) & 1 for i in range(n.bit_length())])      # bit thấp -> cao: [0, 1, 0, 1]
print(n.bit_length(), 0b1010 ^ 0b0101, bin(0b1010 ^ 0b0101).count('1'))

[0, 1, 0, 1]
4 15 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gọi `s` có độ dài `L` và `r = reverse(s)`, tức `r[i] = s[L-1-i]`. Ta muốn biến `s` thành `r`. Vị trí `i` đã khớp (`s[i] == r[i]`) thì khỏi lật; vị trí `i` lệch thì **bắt buộc lật đúng một lần** (lật hai lần là phí). Vậy đáp án là **số vị trí `i` mà `s[i] != s[L-1-i]`**, tức khoảng cách Hamming giữa `s` và `r`.

Tính tay `n = 10`, `s = "1010"`, `r = "0101"`:

```
i      : 0 1 2 3
s      : 1 0 1 0
r      : 0 1 0 1
lệch?  : x x x x   -> 4 vị trí -> 4 lần lật
```

Quan sát thú vị: vị trí `i` lệch khi và chỉ khi vị trí đối xứng `L-1-i` cũng lệch (vì `r[L-1-i] = s[i]`). Do đó các vị trí lệch **đi theo cặp** và đáp án luôn là số **chẵn** `= 2 * (số cặp đối xứng khác nhau)`.

### Bảng tính tay

| n | s | r = reverse(s) | cặp đối xứng khác nhau | đáp án |
|---|---|---|---|---|
| 7 | `111` | `111` | 0 | 0 |
| 10 | `1010` | `0101` | 2 (cặp 0-3, cặp 1-2) | 4 |
| 12 | `1100` | `0011` | 2 | 4 |
| 13 | `1101` | `1011` | 1 (cặp 1-2) | 2 |
| 1 | `1` | `1` | 0 | 0 |
| 6 | `110` | `011` | 1 (cặp 0-2) | 2 |

In [6]:
for n in [7, 10, 12, 13, 1, 6]:
    s = bin(n)[2:]
    r = s[::-1]
    pairs = sum(s[i] != s[len(s) - 1 - i] for i in range(len(s) // 2))
    ham = sum(a != b for a, b in zip(s, r))
    print(f"n={n:>2} s={s:>5} r={r:>5} cặp khác={pairs} Hamming={ham}")
    assert ham == 2 * pairs

n= 7 s=  111 r=  111 cặp khác=0 Hamming=0
n=10 s= 1010 r= 0101 cặp khác=2 Hamming=4
n=12 s= 1100 r= 0011 cặp khác=2 Hamming=4
n=13 s= 1101 r= 1011 cặp khác=1 Hamming=2
n= 1 s=    1 r=    1 cặp khác=0 Hamming=0
n= 6 s=  110 r=  011 cặp khác=1 Hamming=2


## 5. Cách 1 — Brute Force / baseline

Đúng định nghĩa: dựng `s = bin(n)[2:]`, dựng `r = s[::-1]`, rồi đếm số vị trí `i` mà `s[i] != r[i]`.

Nút thắt: nhìn qua thì đã tối ưu ở mức `O(L)` với `L = log2(n)`; chỗ có thể cải thiện là **không cần tạo chuỗi đảo** và chỉ cần xét **một nửa** các vị trí (vì các vị trí lệch đi theo cặp).

In [7]:
class SolutionBrute:
    def minimumFlips(self, n):
        s = bin(n)[2:]
        r = s[::-1]
        flips = 0
        for i in range(len(s)):
            if s[i] != r[i]:
                flips += 1
        return flips

print(SolutionBrute().minimumFlips(7), SolutionBrute().minimumFlips(10))

0 4


**Độ phức tạp:** Time `O(L)` với `L = log2(n) + 1` (tối đa 30), Space `O(L)` cho hai chuỗi.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** vị trí lệch đi theo **cặp đối xứng**: `s[i] != s[L-1-i]` cho cả hai vị trí `i` và `L-1-i`. Do đó chỉ cần đếm số cặp khác nhau (`i < L-1-i`) rồi **nhân 2**.

**Quy tắc "nếu… thì…":**

- Nếu `s[i] == s[L-1-i]` -> cặp khớp, đóng góp `0`.
- Nếu `s[i] != s[L-1-i]` -> cặp lệch, đóng góp `2` lần lật (một lật cho `s[i]`, một lật cho `s[L-1-i]`).
- Ký tự giữa (khi `L` lẻ) luôn khớp với chính nó -> bỏ qua.

**Lưu ý quan trọng:** đích là `reverse(s_gốc)` chứ **không** phải "một chuỗi palindrome bất kỳ". Với palindrome ta chỉ cần lật **một** bit mỗi cặp lệch (đáp án `10 -> 2`), còn ở đây phải lật **cả hai** (đáp án `10 -> 4`).

In [8]:
# Kiểm chứng ý "các vị trí lệch đi theo cặp" bằng vét cạn nhỏ
for n in range(1, 4096):
    s = bin(n)[2:]
    L = len(s)
    for i in range(L):
        assert (s[i] != s[L - 1 - i]) == (s[L - 1 - i] != s[i])
    ham = sum(a != b for a, b in zip(s, s[::-1]))
    assert ham % 2 == 0 and ham == 2 * sum(s[i] != s[L - 1 - i] for i in range(L // 2))
print("số vị trí lệch luôn chẵn và bằng 2 * (số cặp lệch)")

số vị trí lệch luôn chẵn và bằng 2 * (số cặp lệch)


## 7. Các cách tiếp theo

### Cách 2 — Hai con trỏ trên chuỗi, đếm cặp rồi nhân đôi

`i = 0`, `j = L - 1`; khi `i < j`, nếu `s[i] != s[j]` thì `pairs += 1`; `i += 1`, `j -= 1`. Trả `2 * pairs`. Không tạo chuỗi đảo.

In [9]:
class SolutionTwoPointers:
    def minimumFlips(self, n):
        s = bin(n)[2:]
        i, j = 0, len(s) - 1
        pairs = 0
        while i < j:
            if s[i] != s[j]:
                pairs += 1
            i += 1
            j -= 1
        return 2 * pairs

print(SolutionTwoPointers().minimumFlips(7), SolutionTwoPointers().minimumFlips(10))

0 4


**Độ phức tạp:** Time `O(L)` (chỉ `L/2` vòng), Space `O(L)` cho chuỗi `s` (nhưng không có chuỗi phụ thứ hai).

### Cách 3 — Thao tác bit thuần, không dùng chuỗi

Độ dài `L = n.bit_length()`. Với `i` từ `0` đến `L // 2 - 1`, lấy bit thấp `(n >> i) & 1` và bit đối xứng `(n >> (L - 1 - i)) & 1`; nếu khác nhau thì thêm một cặp. Trả `2 * pairs`. Cách này có Space `O(1)`.

In [10]:
class SolutionBits:
    def minimumFlips(self, n):
        L = n.bit_length()
        pairs = 0
        for i in range(L // 2):
            low = (n >> i) & 1
            high = (n >> (L - 1 - i)) & 1
            if low != high:
                pairs += 1
        return 2 * pairs

print(SolutionBits().minimumFlips(7), SolutionBits().minimumFlips(10))

0 4


**Độ phức tạp:** Time `O(L)`, Space `O(1)`.

### Cách 4 — XOR với số "đảo bit" rồi đếm bit `1`

Dựng `rev` là số có chuỗi nhị phân là đảo của `s` (`int(s[::-1], 2)`; số `0` đứng đầu của chuỗi đảo chỉ làm `rev` nhỏ hơn, không ảnh hưởng vì ta XOR trên đúng `L` bit). Khi đó `n ^ rev` có bit `1` đúng ở các vị trí lệch, nên đáp án là `popcount(n ^ rev)`.

In [11]:
class SolutionXor:
    def minimumFlips(self, n):
        s = bin(n)[2:]
        rev = int(s[::-1], 2)               # số có biểu diễn là chuỗi đảo (có thể có 0 đứng đầu)
        return bin(n ^ rev).count('1')

print(SolutionXor().minimumFlips(7), SolutionXor().minimumFlips(10))

0 4


**Độ phức tạp:** Time `O(L)`, Space `O(L)`.

## 8. TRACE từng bước

Trace Cách 2 với `n = 13`, `s = "1101"` (`L = 4`):

| bước | i | j | `s[i]` | `s[j]` | khác nhau? | pairs |
|---|---|---|---|---|---|---|
| 1 | 0 | 3 | 1 | 1 | không | 0 |
| 2 | 1 | 2 | 1 | 0 | có | 1 |
| dừng | 2 | 1 | (i >= j) | | | 1 |

Đáp án `2 * 1 = 2`. Kiểm tra bằng Hamming: `1101` so với `1011` lệch ở vị trí 1 và 2 -> 2.

In [12]:
def trace_two_pointers(n):
    s = bin(n)[2:]
    print(f"n={n} s={s} L={len(s)}")
    i, j, pairs, step = 0, len(s) - 1, 0, 0
    while i < j:
        step += 1
        diff = s[i] != s[j]
        pairs += diff
        print(f"  bước {step}: i={i} j={j} s[i]={s[i]} s[j]={s[j]} khác nhau? {diff} pairs={pairs}")
        i += 1
        j -= 1
    print(f"  dừng i={i} j={j} -> đáp án 2*{pairs} = {2 * pairs}")
    return 2 * pairs

assert trace_two_pointers(13) == 2
assert trace_two_pointers(10) == 4

n=13 s=1101 L=4
  bước 1: i=0 j=3 s[i]=1 s[j]=1 khác nhau? False pairs=0
  bước 2: i=1 j=2 s[i]=1 s[j]=0 khác nhau? True pairs=1
  dừng i=2 j=1 -> đáp án 2*1 = 2
n=10 s=1010 L=4
  bước 1: i=0 j=3 s[i]=1 s[j]=0 khác nhau? True pairs=1
  bước 2: i=1 j=2 s[i]=0 s[j]=1 khác nhau? True pairs=2
  dừng i=2 j=1 -> đáp án 2*2 = 4


In [13]:
def trace_bits(n):
    L = n.bit_length()
    print(f"n={n} nhị phân={bin(n)[2:]} L={L}")
    pairs = 0
    for i in range(L // 2):
        low, high = (n >> i) & 1, (n >> (L - 1 - i)) & 1
        pairs += low != high
        print(f"  i={i}: bit thấp thứ {i} = {low}, bit đối xứng thứ {L - 1 - i} = {high}, pairs={pairs}")
    return 2 * pairs

assert trace_bits(12) == 4 and trace_bits(7) == 0

n=12 nhị phân=1100 L=4
  i=0: bit thấp thứ 0 = 0, bit đối xứng thứ 3 = 1, pairs=1
  i=1: bit thấp thứ 1 = 0, bit đối xứng thứ 2 = 1, pairs=2
n=7 nhị phân=111 L=3
  i=0: bit thấp thứ 0 = 1, bit đối xứng thứ 2 = 1, pairs=0


## 9. Code nộp LeetCode

In [14]:
from typing import List, Optional

class Solution:
    def minimumFlips(self, n: int) -> int:
        s = bin(n)[2:]
        i, j = 0, len(s) - 1
        pairs = 0
        while i < j:
            if s[i] != s[j]:
                pairs += 1               # mỗi cặp đối xứng khác nhau cần lật cả hai bit
            i += 1
            j -= 1
        return 2 * pairs

## 10. Vì sao thuật toán đúng

- **Đích cố định:** sau các lần lật, `s` phải bằng `r = reverse(s_gốc)`. Vị trí `i` cần giá trị `r[i] = s_gốc[L-1-i]`.
- **Mỗi vị trí độc lập:** lật vị trí `i` không ảnh hưởng vị trí khác. Nếu `s[i] == r[i]` thì không cần lật; nếu khác thì lật **đúng một lần** (lật hai lần trả về giá trị cũ). Do đó số lần lật tối thiểu là số vị trí `i` với `s[i] != r[i]`.
- **Ghép cặp:** `s[i] != r[i] = s[L-1-i]` tương đương `s[L-1-i] != s[i] = r[L-1-i]`, nên vị trí `i` lệch khi và chỉ khi `L-1-i` lệch. Mỗi cặp `(i, L-1-i)` với `i < L-1-i` đóng góp `0` hoặc `2`; vị trí giữa (nếu có) đóng góp `0`.
- **Các cách khác nhau cùng đáp án:** cách 1 đếm trực tiếp, cách 2 và 3 đếm theo cặp rồi nhân 2, cách 4 dùng XOR (bit `1` đúng ở vị trí lệch). Mục 12 kiểm tra chúng khớp nhau và khớp một oracle vét cạn theo mặt nạ lật.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| So `s` với `s[::-1]` | `O(L)` | `O(L)` |
| Hai con trỏ | `O(L)` | `O(L)` |
| Thao tác bit | `O(L)` | `O(1)` |
| XOR + đếm bit | `O(L)` | `O(L)` |

`L = ⌊log2 n⌋ + 1 <= 30` (vì `n <= 10^9 < 2^30`). Mọi cách đều gần như hằng số; khác biệt chủ yếu về phong cách và bộ nhớ.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, `n = 10^9`, lũy thừa của 2, số palindrome nhị phân) và so sánh **tất cả cách** với một **oracle vét cạn**: thử mọi mặt nạ lật `mask` và lấy mặt nạ có ít bit nhất sao cho `s xor mask == reverse(s)`.

In [15]:
def oracle(n):
    s = bin(n)[2:]
    L = len(s)
    target = int(s[::-1].rjust(L, '0'), 2)
    best = None
    for mask in range(1 << L):                    # mọi tập vị trí được lật
        if n ^ mask == target:
            c = bin(mask).count('1')
            best = c if best is None else min(best, c)
    return best

impls = [SolutionBrute(), SolutionTwoPointers(), SolutionBits(), SolutionXor(), Solution()]

cases = [(7, 0), (10, 4), (1, 0), (2, 2), (3, 0), (4, 2), (5, 0), (6, 2), (12, 4), (13, 2), (8, 2), (9, 0)]
for n, exp in cases:
    assert 1 <= n <= 10 ** 9
    assert oracle(n) == exp, (n, oracle(n), exp)
    for impl in impls:
        got = impl.minimumFlips(n)
        assert got == exp, (type(impl).__name__, n, got, exp)
print("ví dụ + edge OK (khớp cả oracle vét cạn):", len(cases), "ca")

ví dụ + edge OK (khớp cả oracle vét cạn): 12 ca


In [16]:
# Vét cạn toàn bộ n nhỏ: so sánh 5 cách với oracle mặt nạ lật
for n in range(1, 2049):
    exp = oracle(n)
    assert all(impl.minimumFlips(n) == exp for impl in impls), n
print("n = 1..2048: 5 cách khớp oracle")

# Palindrome nhị phân -> 0; lũy thừa của 2 (s = 1 rồi các số 0): chỉ cặp đầu-cuối lệch -> 2 (n = 1 thì 0)
for n in range(1, 5000):
    s = bin(n)[2:]
    if s == s[::-1]:
        assert Solution().minimumFlips(n) == 0
for k in range(1, 30):
    assert Solution().minimumFlips(1 << k) == 2, k
assert Solution().minimumFlips(1) == 0
print("palindrome nhị phân -> 0; lũy thừa của 2 (>= 2) -> 2 OK")

n = 1..2048: 5 cách khớp oracle
palindrome nhị phân -> 0; lũy thừa của 2 (>= 2) -> 2 OK


In [17]:
import random
random.seed(3750)
for _ in range(2000):
    n = random.randint(1, 10 ** 9)
    outs = [impl.minimumFlips(n) for impl in impls]
    assert all(o == outs[0] for o in outs), (n, outs)
    assert outs[0] % 2 == 0 and 0 <= outs[0] <= 2 * (n.bit_length() // 2)
for n in [10 ** 9, 2 ** 29, 2 ** 29 + 1]:
    assert all(impl.minimumFlips(n) == impls[0].minimumFlips(n) for impl in impls)
print("2000 test ngẫu nhiên (seed=3750) OK: 5 cách giống nhau, đáp án chẵn và <= 2*(L//2)")

2000 test ngẫu nhiên (seed=3750) OK: 5 cách giống nhau, đáp án chẵn và <= 2*(L//2)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhầm với "biến `s` thành palindrome"

Nếu đích là một palindrome bất kỳ thì mỗi cặp lệch chỉ cần lật **một** bit. Nhưng đề yêu cầu `s` bằng **bản đảo của chính nó lúc đầu**, nên phải lật cả hai bit của mỗi cặp lệch.

In [18]:
def wrong_palindrome_flips(n):
    s = bin(n)[2:]
    return sum(s[i] != s[len(s) - 1 - i] for i in range(len(s) // 2))     # SAI: thiếu nhân 2

print(wrong_palindrome_flips(10), "!=", Solution().minimumFlips(10))
assert wrong_palindrome_flips(10) == 2 and Solution().minimumFlips(10) == 4

2 != 4


### Lỗi 2: đếm mỗi cặp hai lần rồi nhân đôi thêm lần nữa

Duyệt **toàn bộ** `i` trong `range(L)` đã đếm mỗi cặp hai lần; nhân thêm `2` sẽ ra gấp bốn lần số cặp.

In [19]:
def wrong_double_count(n):
    s = bin(n)[2:]
    cnt = sum(s[i] != s[len(s) - 1 - i] for i in range(len(s)))          # đã đếm hai lần mỗi cặp
    return 2 * cnt                                                       # SAI: nhân đôi lần nữa

print(wrong_double_count(10), "!=", Solution().minimumFlips(10))
assert wrong_double_count(10) == 8 and Solution().minimumFlips(10) == 4

8 != 4


### Lỗi 3: quên bỏ tiền tố `'0b'`

`bin(7)` là `'0b111'`; đảo ngược thành `'111b0'`. Các ký tự `'0'`, `'b'` được đưa vào so sánh nên số lần lật bị tính sai.

In [20]:
def wrong_prefix(n):
    s = bin(n)                                   # SAI: thiếu [2:]
    return sum(a != b for a, b in zip(s, s[::-1]))

print(wrong_prefix(7), "!=", Solution().minimumFlips(7))
assert wrong_prefix(7) == 4 and Solution().minimumFlips(7) == 0

4 != 0


### Lỗi 4: dùng hiệu hai số thay vì số bit khác nhau

`abs(n - rev)` là chênh lệch **giá trị**, còn đề hỏi số **bit** cần lật.

In [21]:
def wrong_numeric_difference(n):
    s = bin(n)[2:]
    return abs(n - int(s[::-1], 2))              # SAI: |10 - 5| = 5

print(wrong_numeric_difference(10), "!=", Solution().minimumFlips(10))
assert wrong_numeric_difference(10) == 5 and Solution().minimumFlips(10) == 4

5 != 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số lần lật tối thiểu để biến chuỗi A thành chuỗi B cùng độ dài" = **khoảng cách Hamming** giữa A và B.
- Khi B là **đảo của A**, các vị trí lệch đi theo cặp -> đáp án chẵn, bằng `2 * (số cặp đối xứng lệch)`.
- Phân biệt: biến thành **palindrome** (mỗi cặp lệch chỉ cần 1 lật) với biến thành **reverse cố định** (mỗi cặp lệch cần 2 lật).
- `bin(n)[2:]` cho chuỗi nhị phân; `n.bit_length()` cho độ dài; nhớ bỏ `'0b'`.
- XOR cho bit `1` ở nơi khác nhau; `bin(x).count('1')` là popcount.
- Tự kiểm tra: số palindrome nhị phân (`1, 3, 5, 7, 9, 15, ...`) phải ra `0`.

## 15. Biến thể của bài

1. **Minimum Bit Flips to Convert Number** (LeetCode 2220): số bit cần lật để `start` thành `goal` = `popcount(start ^ goal)`.
2. **Hamming Distance** (461): số vị trí bit khác nhau của hai số.
3. **Reverse Bits** (190): đảo bit của số 32 bit.
4. **Binary Number with Alternating Bits** (693) và **Binary Gap** (868): xét các bit liền kề.
5. **Palindrome Number** (9) và **Valid Palindrome II** (680): so sánh đầu-cuối bằng hai con trỏ.
6. **Minimum number of flips để `s` thành palindrome** (nhiều đề tương tự): đáp án là số cặp lệch.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| `s` so với `s[::-1]` | đếm vị trí khác nhau | `O(L)` | `O(L)` | Dễ hiểu nhất, bám định nghĩa |
| Hai con trỏ | đếm cặp lệch rồi nhân 2 | `O(L)` | `O(L)` | Nộp bài, bài kiểu palindrome |
| Thao tác bit | so bit thấp với bit đối xứng | `O(L)` | `O(1)` | Không muốn dùng chuỗi |
| XOR + popcount | `n ^ rev` rồi đếm bit 1 | `O(L)` | `O(L)` | Gọn, dùng được cho bài "bit flips" |

Nộp cách hai con trỏ (nói rõ được vì sao nhân 2).

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minimumFlips(20)` ra gì? (`20 = 10100` nhị phân)

**Bài 2 (tính tay):** với `n = 6` (`110`) và `n = 13` (`1101`), viết `s`, `r` rồi đếm vị trí lệch.

**Bài 3 (điền TODO):** hoàn thành `flips_to_palindrome(n)` trả về số lần lật ít nhất để chuỗi nhị phân của `n` thành **một palindrome bất kỳ** (đáp án của biến thể).

**Bài 4 (tự viết lại):** viết `minimum_flips_xor(n)` chỉ dùng toán tử bit (`>>`, `&`, `|`, `^`) để dựng số đảo bit `rev` mà **không** dùng chuỗi, rồi trả `popcount(n ^ rev)`.

**Bài 5 (mini challenge):** viết `count_zero_answer(limit)` đếm các `n` trong `1..limit` có đáp án bằng `0` (tức palindrome nhị phân); kiểm tra `count_zero_answer(15) == 6`.

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minimumFlips(20))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def flips_to_palindrome_todo(n):
    # TODO: đếm số cặp đối xứng khác nhau của bin(n)[2:]
    return 0

2


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `10100` đảo là `00101`, so từng vị trí (hai vị trí khác nhau). Bài 2: `110` đảo là `011`; `1101` đảo là `1011`.

**Mức 2 (vừa):** Bài 3: chính là "số cặp lệch" **không nhân 2**. Bài 4: dựng `rev` bằng vòng `rev = (rev << 1) | (x & 1); x >>= 1` lặp `L` lần. Bài 5: đáp án bằng `0` khi chuỗi nhị phân là palindrome; thử liệt kê `bin(n)[2:]` cho `n = 1..15`.

**Mức 3 (gần đáp án):** Bài 1: vị trí 0 (`1` vs `0`), vị trí 2 (`1` vs `1`), vị trí 4 (`0` vs `1`), vị trí 1 và 3 (`0` vs `0`): lệch ở 0 và 4 -> đáp án `2`. Bài 2: `n = 6 -> 2`, `n = 13 -> 2`. Bài 5: các `n <= 15` có nhị phân là palindrome là `1, 3, 5, 7, 9, 15`, tức `6` số.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1: 20 = 10100, đảo 00101 -> lệch ở vị trí 0 và 4 (một cặp) -> 2
assert Solution().minimumFlips(20) == 2

# Bài 2
assert Solution().minimumFlips(6) == 2 and Solution().minimumFlips(13) == 2

# Bài 3
def flips_to_palindrome(n):
    s = bin(n)[2:]
    return sum(s[i] != s[len(s) - 1 - i] for i in range(len(s) // 2))
assert flips_to_palindrome(10) == 2 and flips_to_palindrome(7) == 0
assert all(2 * flips_to_palindrome(n) == Solution().minimumFlips(n) for n in range(1, 3000))

# Bài 4
def minimum_flips_xor(n):
    L, x, rev = n.bit_length(), n, 0
    for _ in range(L):
        rev = (rev << 1) | (x & 1)       # lấy bit thấp của x đẩy vào rev -> đảo bit
        x >>= 1
    d, cnt = n ^ rev, 0
    while d:
        cnt += d & 1
        d >>= 1
    return cnt
assert minimum_flips_xor(10) == 4 and minimum_flips_xor(7) == 0
assert all(minimum_flips_xor(n) == Solution().minimumFlips(n) for n in range(1, 3000))

# Bài 5
def count_zero_answer(limit):
    return sum(1 for n in range(1, limit + 1) if Solution().minimumFlips(n) == 0)
assert count_zero_answer(15) == 6        # 1, 3, 5, 7, 9, 15
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lật bit để chuỗi thành đích cố định" (khoảng cách Hamming):**

```python
s = bin(n)[2:]
i, j, pairs = 0, len(s) - 1, 0
while i < j:
    if s[i] != s[j]:
        pairs += 1
    i += 1
    j -= 1
return 2 * pairs        # đích là reverse(s) -> lật cả hai bit của cặp lệch
```

**Checklist trước khi submit:**

- [ ] Đã dùng `bin(n)[2:]` (bỏ `'0b'`)?
- [ ] Hiểu đích là `reverse(s)` (nhân 2), không phải palindrome tùy ý?
- [ ] Hai con trỏ dừng khi `i >= j`; ký tự giữa không tính?
- [ ] Thử `n = 1`, `7` (palindrome -> 0), `10` (-> 4), `10^9`?
- [ ] Đáp án luôn chẵn; nếu ra số lẻ chắc chắn có lỗi?
- [ ] Chữ ký đúng `minimumFlips(self, n: int) -> int`?